# Kaggle runner — thesis experiments

Runs `scripts/train.py` from a **tagged commit** of the GitHub repository, so every result is tied to exact code.

1. Settings → Accelerator: GPU. Settings → Internet: on.
2. Set `REPO_TAG` and the list of `RUNS` below.
3. Run all. Outputs land in `/kaggle/working/runs/…`; save the notebook version, then download
   `runs.zip` (or attach the output as a Kaggle Dataset).

Each run folder contains `generated_returns.npy`, `history.json`, `preprocessing.json`, weights and `manifest.json`.

In [ ]:
REPO_URL = "https://github.com/annagrenz/Generation-Of-Financial-Time-Series-Data"
REPO_TAG = "main"   # replace by a tag such as "experiments-v1" for the final runs

# (config, fold, seed, pilot)
RUNS = [
    ("config/mmd_sig.yaml",   4, 0, True),
    ("config/diffusion.yaml", 0, 0, True),
]

In [ ]:
%%bash -s "$REPO_URL" "$REPO_TAG"
rm -rf /kaggle/working/repo
git clone --depth 1 --branch "$2" "$1" /kaggle/working/repo
cd /kaggle/working/repo && git log -1 --format='%H %s'
pip install -q -r requirements.txt

In [ ]:
import subprocess, sys, os
os.chdir("/kaggle/working/repo")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
for cfg, fold, seed, pilot in RUNS:
    cmd = [sys.executable, "scripts/train.py", "--config", cfg, "--fold", str(fold), "--seed", str(seed),
           "--out-root", "/kaggle/working/runs"] + (["--pilot"] if pilot else [])
    print(">>", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)

In [ ]:
%%bash
cd /kaggle/working && zip -qr runs.zip runs && ls -lh runs.zip
find runs -name manifest.json | head